# Session 2 · Part 2 — Advanced Distributed Algorithms

**15:45 – 16:15**

---

Everything up to here was one primitive at a time: a teleported qubit, a Bell
pair, a non-local CNOT. This notebook puts them together into two real
algorithms, on three vQPUs, **one data qubit per node**.

| Exercise | Build | Depends on |
|---|---|---|
| **8** | 3-qubit inverse QFT, distributed | `expose` / `unexpose` from notebook 03 |
| **9** *(optional)* | distributed quantum phase estimation | exercise 8, unchanged, as a subroutine |

Exercise 8 builds a subroutine. Exercise 9 is the algorithm that subroutine
belongs to — and the payoff is that it reuses exercise 8's code *verbatim*, on a
rank that now holds two data qubits and arrives entangled rather than in a
product state. A distributed collective that composes is not a given; this one
does.

> **You need the vQPU family from notebook 04.** Section 0 attaches to it, and
> raises it only if it is not there. `qdrop` waits until the end of this
> notebook.

## 0 · Setup

The same two lines as notebook 04 — the venv's `bin/` onto `PATH` so that
`!netqmpi` resolves, and `$HOME` onto `sys.path` so that CUNQA imports — then
attach to the family.

In [ ]:
import os
import sys
from pathlib import Path

# The venv's bin/, so that `!netqmpi ...` resolves in a shell cell.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

# CUNQA lives in $HOME — the same line every NetQMPI CUNQA adapter starts with.
sys.path.append(os.getenv("HOME"))

from cunqa.qpu import get_QPUs, qdrop, qraise

WORK = Path("notebook_apps")
WORK.mkdir(exist_ok=True)

FAMILY = "netqmpi_notebook"   # the family raised in notebook 04
SIZE = 3                      # nodes / ranks / vQPUs

print("python :", sys.executable)
!which netqmpi qraise qdrop

In [ ]:
# The family from notebook 04, still up. Raised here only if it is not.
qpus = get_QPUs(co_located=True, family=FAMILY) or []

if len(qpus) < SIZE:
    qraise(SIZE, "02:00:00",
           simulator="Munich",
           co_located=True,
           quantum_comm=True,
           backend=str(WORK / "notebook_vqpu.json"),
           family=FAMILY)
    qpus = get_QPUs(co_located=True, family=FAMILY) or []

for qpu in qpus:
    print(f"{qpu.id}  family={qpu.family}  qubits={qpu.backend.get('num_qubits')}")
assert len(qpus) >= SIZE, f"need {SIZE} vQPUs, found {len(qpus)}"

---
## 1 · Telegates, and why a QFT wants them

A QFT is a ladder of Hadamards and **controlled** phase rotations, and every
qubit is a control for every qubit below it. Split those qubits across nodes and
most of the rotations cross a cut.

Teleporting a control across and back costs two transfers each way, and — worse —
drags along whatever the qubit is already entangled with. So the QFT does not
want to *move* qubits. It wants to **lend** them: the control stays where it is
and is made available to the others through a shared GHZ state. That is a
**telegate**, and it is what `expose` / `unexpose` do:

```python
control = comm.expose(circuit, qubit, ranks, root=r)   # r lends `qubit` to `ranks`
...                                                    # everyone rotates locally
comm.unexpose(circuit, ranks, root=r)                  # given back untouched
```

Three properties matter for what follows, all of them met in notebook 03 and all
of them load-bearing here:

1. **They are collective, like `MPI_Bcast`.** Every rank in `[root] + ranks`
   calls them, in the same order. Ranks outside the window get `None` back and
   consume nothing — which is why the calls sit *outside* the `if rank ==` blocks.
2. **`expose` returns the index to use as the control** — the root's own qubit,
   a fresh comm qubit on each receiver — so the gate is written exactly like a
   local one.
3. **Windows nest like scopes.** Two can be open at once; the inner one must
   close first. That is not a style rule, it is the reason the vQPU raised in
   notebook 04 needed two comm qubits rather than one.

---
## Exercise 8 — a 3-qubit inverse QFT across three QPUs

`examples/netqmpi/5_qft_expose.py` builds the forward transform:

```
H(q0) · CP(q1→q0, π/2) · CP(q2→q0, π/4) · H(q1) · CP(q2→q1, π/2) · H(q2)
```

**Write the inverse**: the same gates back to front, every angle negated.

```
H(q2) · CP(q2→q1, -π/2) · H(q1) · CP(q2→q0, -π/4) · CP(q1→q0, -π/2) · H(q0)
```

Two consequences, and they *are* the exercise:

* `q2`'s `H` comes **first** now, before it is lent to anybody.
* `q1` is a receiver and *then* a lender, so its window opens inside `q2`'s — and
  therefore has to close before it.

**How you will know it worked.** `QFT|x>` is a product state, so each rank can
prepare its own factor locally with no communication at all (this part is already
written for you): rank *j* holds `(|0> + e^{2πi·2^j·x/8}|1>)/√2`. Undo the
transform and every rank reads one bit of `x` on all 1024 shots — rank 0 the most
significant. `report` at the top of the file does that reading and prints the
verdict, so the run tells you itself whether it worked.

The value of `x` travels in the environment: the launcher passes no arguments
through to the program.

In [ ]:
%%writefile notebook_apps/ex8_iqft.py
"""
Exercise 8: the 3-qubit inverse QFT, one data qubit per rank.

Each rank prepares its factor of QFT|x> locally, the three of them undo the
transform together, and each reads back one bit of x — rank 0 the most
significant, rank 2 the least.

Run with::

    QFT_X=5 netqmpi -n 3 notebook_apps/ex8_iqft.py --cunqa --shots 1024
"""
import os

import numpy as np

from netqmpi.sdk.environment import Environment

#: The value to encode. The launcher passes no arguments to the program, so a
#: parameter of an SPMD run travels through the environment.
X = int(os.environ.get("QFT_X", 5))

def report(results):
    """
    Print each rank's marginal and the integer the three of them spell out.

    CUNQA joins one bitstring per classical register with spaces, in the order
    the registers were added, so the user's own single-bit register comes first
    and the two NetQMPI protocol bits after it. Rank 0 holds the most
    significant bit.

    These are *marginals*: one rank's counts say nothing about another's. That
    is enough here only because the state going in makes every marginal a
    delta, and ``certain`` reports whether that actually happened.

    Args:
        results: ``{rank: counts}``, as the last rank out of the block sees it.

    Returns:
        ``(value, certain)`` — the integer read off, and whether every rank
        agreed with itself on all shots.
    """
    bits, certain = "", True
    for rank in sorted(results):
        counts = results[rank]
        shots = sum(counts.values())
        marginal = {"0": 0, "1": 0}
        for key, n in counts.items():
            marginal[key.split()[0][-1]] += n
        bit = max(marginal, key=marginal.get)
        certain &= marginal[bit] == shots
        bits += bit
        print(f"  rank_{rank}: P(0)={marginal['0'] / shots:5.3f}  "
              f"P(1)={marginal['1'] / shots:5.3f}   -> {bit}")
    return int(bits, 2), certain


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    with comm:
        circuit = env.create_circuit(num_qubits=1, num_clbits=1)

        # --- prepare this rank's factor of QFT|X> -----------------------
        # rank j holds (|0> + e^{2 pi i 2^j X/8} |1>)/sqrt(2). rz is a phase
        # gate up to e^{-i theta/2}, and since every rank applies one
        # unconditionally those factors collapse into one global phase.
        circuit.h(0)
        circuit.rz(2 * np.pi * (2 ** rank) * X / 8, 0)

        # --- undo the transform ----------------------------------------
        # The gates to produce, in this order:
        #   H(q2) CP(q2->q1,-pi/2) H(q1) CP(q2->q0,-pi/4) CP(q1->q0,-pi/2) H(q0)
        #
        # TODO 1: rank 2's H, before it lends its qubit to anybody.
        # TODO 2: open the window in which rank 2 lends qubit 0 to ranks 0 and 1.
        # TODO 3: rank 1 rotates on the borrowed control, then does its own H.
        # TODO 4: open the window in which rank 1 lends qubit 0 to rank 0.
        # TODO 5: rank 0 does both of its rotations, then its own H.
        # TODO 6: close the two windows — innermost (root 1) first.
        raise NotImplementedError("exercise 8 — delete this line and fill in the TODOs")

        circuit.measure(0, 0)

    # Every rank's circuit is submitted together when the last one leaves the
    # block, so only that rank sees the results — and it sees all of them.
    if comm.results:
        value, certain = report(comm.results)
        print(f"x = {X} -> read {value}   deterministic={certain}   "
              f"{'ok' if value == X and certain else 'WRONG'}")

<details>
<summary><b>Hints</b></summary>

* Two windows are open at once, so keep both handles:
  `c2 = comm.expose(circuit, 0, [0, 1], root=2)` and
  `c1 = comm.expose(circuit, 0, [0], root=1)`.
* Every rank calls `expose(..., root=2)`, rank 2 included — it is the root, and
  gets its own qubit 0 back.
* Windows nest like scopes: `unexpose(..., root=1)` before
  `unexpose(..., root=2)`.
* `circuit.cp(control, target, theta)` — control first. On rank 1 the control is
  `c2`, the target its own qubit 0.
* Rank 1's `H` goes *between* the rotation it receives and the window it opens:
  rank 0 must borrow the transformed qubit.
* The prep uses `rz`, not `P` — the SDK has no single-qubit phase gate. They
  differ by `e^{-iθ/2}`, and since every rank applies one unconditionally those
  collapse into one unobservable global phase.

</details>

In [ ]:
!QFT_X=5 netqmpi -n 3 notebook_apps/ex8_iqft.py --cunqa --shots 1024

### Check

`101` is 5, rank 0 first. Every `P(0)`/`P(1)` pair should be a clean `1.000` /
`0.000`, and the last line should end in `ok`.

`deterministic` is worth more than the value itself. It guards the one assumption
that comes from the counts *format* rather than the physics: the protocol bits
are coin flips, so if `report` had picked one of those by mistake nothing would
be deterministic, and you would get `deterministic=False` rather than a plausible
wrong answer.

One value could be luck. Eight cannot — each is a separate three-vQPU run with a
different product state going in. Takes about a minute.

In [ ]:
!for x in 0 1 2 3 4 5 6 7; do QFT_X=$x netqmpi -n 3 notebook_apps/ex8_iqft.py --cunqa --shots 1024 | tail -1; done

> **Why the vQPU needed two comm qubits.** Rank 0 sits inside both windows at
> once, so it holds two borrowed controls simultaneously. Close rank 2's window
> earlier and one comm qubit would do — but the circuit would no longer be the
> inverse QFT. **Window lifetime is the comm-qubit budget.**
>
> Note also what a window costs: *one* GHZ state, however many rotations happen
> inside it. Rank 2's control is used twice, by two different ranks, for the
> price of lending it once. Teleporting `q2` back and forth would have meant four
> teledata blocks.

---
## Exercise 9 *(optional)* — distributed phase estimation

Exercise 8 built a subroutine. This is the algorithm it belongs to.

Phase estimation with `U = P(2πφ)`, whose eigenstate is `|1>`:

1. counting qubits into `|+>`, the eigenstate on its own qubit;
2. `controlled-U^(2^j)` from counting qubit *j* — one `cp` gate — kicks the phase
   back, leaving qubit *j* in `(|0> + e^{2πi·2^j·φ}|1>)/√2`;
3. inverse-QFT the counting register and measure.

Step 2 leaves exactly the product state you prepared by hand in exercise 8, so
step 3 *is* exercise 8 — `inverse_qft` is already in the file, copied over
unchanged.

The layout, and the first program of the day in which the ranks are **not
symmetric**:

```
rank 0:  q0 = counting bit 0 (most significant),  q1 = |psi>
rank 1:  q0 = counting bit 1
rank 2:  q0 = counting bit 2 (least significant)
```

Every `controlled-U^(2^j)` has its control on rank *j* and its target on rank 0:
local for rank 0, an `expose` window for the other two. Those windows do not
overlap, so they hand the same comm qubit back and forth.

Write steps 1 and 2.

In [ ]:
%%writefile notebook_apps/ex9_qpe.py
"""
Exercise 9: distributed quantum phase estimation on 3 nodes.

U = P(2 pi phi) with eigenstate |1>, three counting qubits one per rank. Rank 0
holds the eigenstate as a second data qubit; the other ranks reach it by
exposing their counting qubit as a control.

Run with::

    QPE_PHI=0.625 netqmpi -n 3 notebook_apps/ex9_qpe.py --cunqa --shots 1024
"""
import os

import numpy as np

from netqmpi.sdk.environment import Environment

#: Phase to estimate. Exactly representable in 3 bits when it is a multiple of 1/8.
PHI = float(os.environ.get("QPE_PHI", 5 / 8))

#: Rank 0's second data qubit, holding the eigenstate.
EIGEN = 1

def report(results):
    """
    Print each rank's marginal and the integer the three of them spell out.

    CUNQA joins one bitstring per classical register with spaces, in the order
    the registers were added, so the user's own single-bit register comes first
    and the two NetQMPI protocol bits after it. Rank 0 holds the most
    significant bit.

    These are *marginals*: one rank's counts say nothing about another's. That
    is enough here only because the state going in makes every marginal a
    delta, and ``certain`` reports whether that actually happened.

    Args:
        results: ``{rank: counts}``, as the last rank out of the block sees it.

    Returns:
        ``(value, certain)`` — the integer read off, and whether every rank
        agreed with itself on all shots.
    """
    bits, certain = "", True
    for rank in sorted(results):
        counts = results[rank]
        shots = sum(counts.values())
        marginal = {"0": 0, "1": 0}
        for key, n in counts.items():
            marginal[key.split()[0][-1]] += n
        bit = max(marginal, key=marginal.get)
        certain &= marginal[bit] == shots
        bits += bit
        print(f"  rank_{rank}: P(0)={marginal['0'] / shots:5.3f}  "
              f"P(1)={marginal['1'] / shots:5.3f}   -> {bit}")
    return int(bits, 2), certain


def inverse_qft(comm, circuit, rank):
    """The 3-qubit inverse QFT of exercise 8, one counting qubit per rank."""
    # Rank 2 is done being transformed before it becomes a control, so its H
    # comes first — the mirror of the forward transform, where it came last.
    if rank == 2:
        circuit.h(0)

    # Rank 2 lends its qubit to both other ranks: they each need it as a
    # control. The window stays open until the very end.
    control_2 = comm.expose(circuit, 0, [0, 1], root=2)

    if rank == 1:
        circuit.cp(control_2, 0, -np.pi / 2)
        circuit.h(0)

    # Only now is rank 1's qubit the one rank 0 needs as a control, so its
    # window opens inside rank 2's and will have to close before it.
    control_1 = comm.expose(circuit, 0, [0], root=1)

    if rank == 0:
        circuit.cp(control_2, 0, -np.pi / 4)
        circuit.cp(control_1, 0, -np.pi / 2)
        circuit.h(0)

    comm.unexpose(circuit, [0], root=1)
    comm.unexpose(circuit, [0, 1], root=2)


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    with comm:
        # Rank 0 needs a second data qubit for the eigenstate.
        circuit = env.create_circuit(
            num_qubits=2 if rank == 0 else 1, num_clbits=1)

        # TODO 1: rank 0 prepares the eigenstate |1> on qubit EIGEN.
        # TODO 2: every rank puts its counting qubit (qubit 0) into |+>.
        # TODO 3: rank 0's own kickback — a local cp(0, EIGEN, 2*pi*PHI).
        # TODO 4: for j in (1, 2): rank j exposes its counting qubit to rank 0,
        #         rank 0 applies cp(control, EIGEN, 2*pi*(2**j)*PHI), close it.
        raise NotImplementedError("exercise 9 — delete this line and fill in the TODOs")

        # --- read the phase out of the counting register ----------------
        inverse_qft(comm, circuit, rank)

        circuit.measure(0, 0)

    if comm.results:
        value, certain = report(comm.results)
        eighths = PHI * 8
        if abs(eighths - round(eighths)) < 1e-9:
            expected = round(eighths) % 8
            print(f"phi = {PHI:.4f} = {expected}/8 -> read {value}/8   "
                  f"deterministic={certain}   "
                  f"{'ok' if value == expected and certain else 'WRONG'}")
        else:
            print(f"phi = {PHI:.4f} -> read {value}/8 = {value / 8:.4f}   "
                  f"deterministic={certain}   (not a multiple of 1/8)")

<details>
<summary><b>Hints</b></summary>

* `circuit.x(EIGEN)` on rank 0 only, before any `cp` touches it.
* `circuit.h(0)` is unconditional — every counting qubit starts in `|+>`.
* Rank 0's own kickback needs no window: `circuit.cp(0, EIGEN, 2 * np.pi * PHI)`.
* The rest is three lines, with the collectives *outside* the `if`:

  ```python
  for j in (1, 2):
      control = comm.expose(circuit, 0, [0], root=j)
      if rank == 0:
          circuit.cp(control, EIGEN, 2 * np.pi * (2 ** j) * PHI)
      comm.unexpose(circuit, [0], root=j)
  ```

* Angles are `2π·2^j·φ`, not `2π·φ/2^j`. Rank 0 is the most significant bit, so
  it gets the *smallest* rotation — which is the ordering the inverse QFT
  expects, and why the two halves fit.

</details>

### Check — a phase it can represent exactly

`φ = 5/8` is a multiple of `1/8`, so three bits hold it exactly and the counting
register is deterministic: `101`, every shot. Then all eight.

In [ ]:
!QPE_PHI=0.625 netqmpi -n 3 notebook_apps/ex9_qpe.py --cunqa --shots 1024

In [ ]:
!for phi in 0 0.125 0.25 0.375 0.5 0.625 0.75 0.875; do QPE_PHI=$phi netqmpi -n 3 notebook_apps/ex9_qpe.py --cunqa --shots 1024 | tail -1; done

### Check — a phase it cannot

`φ = 1/3` falls between `2/8 = 0.25` and `3/8 = 0.375`, so the register straddles
it. Read this one carefully: three **marginals** come back, and nothing in them
says which shot of rank 0 went with which of rank 2. For the exact phases that
was fine — a product of deltas *is* the joint distribution. Here it is not, and
the histogram over `x` simply is not in this data.

`2/8 = 010` and `3/8 = 011` agree on the first two bits and differ in the third,
which is exactly what you should see: ranks 0 and 1 nearly certain, rank 2 split
near 50/50, and `deterministic=False` — the honest signature.

In [ ]:
!QPE_PHI=0.3333333333333333 netqmpi -n 3 notebook_apps/ex9_qpe.py --cunqa --shots 1024

> **The composition is the result.** `inverse_qft` was copied out of exercise 8
> unchanged and kept working even though rank 0 now holds two data qubits and
> arrives entangled rather than in a product state. A collective is written
> against ranks and qubit indices, so it does not care what happened before it.
>
> And note the budget: five windows over the whole program, still `[2, 2]`. The
> kickback windows open and close one at a time; only the inverse QFT's nested
> pair is ever held at once. **Peak concurrency, not total count, is what you
> pay for.**

---
## 2 · Release the resources

The family has been up since notebook 04, which is why every run in this session
attached instantly and paid no SLURM cost. It is still a reservation. Drop it.

In [ ]:
qdrop(FAMILY)
print(f"dropped {FAMILY}")

In [ ]:
!squeue

---
## Where to go next

**16:15 – 16:30 is Q&A** — bring what broke.

* `examples/netqmpi/` — `3_scatter.py` and `4_gather.py`, the rooted collectives
  that *move* qubits rather than lend them.
* `examples/netqmpi/frequent_errors/` — a dozen programs meant to fail, one mode
  each, with a `run_all.py` that needs no vQPU. The fastest way to learn the
  diagnostics.
* `netqmpi/papers/Emulating_NetQMPI_applications_with_CUNQA.pdf` — the
  architecture underneath all of this.

Two to try on your own:

1. Redo exercise 8 with **teledata** instead of telegates: move `q1` and `q2` to
   rank 0, transform, move them back. Count the `gen_ent` requests in each.
2. Take exercise 9 to four nodes. Work out what to raise *before* raising it —
   `n` counting qubits over `n` nodes means `n(n-1)/2` cross-node rotations and
   `n-1` comm qubits held at once on rank 0.

---

### What you built today

| | |
|---|---|
| 01 | a distributed system is an address-space boundary, and messages cost `α + βn` |
| 02 | SPMD, point-to-point, collectives — and the copy / move / combine split |
| 03 | the same shapes over qubits, with copying forbidden and `expose` in place of `Bcast` |
| 04 | what a vQPU is, what a family costs, and what `qsend` expands into |
| 05 | two real algorithms on three nodes, one composed out of the other |